## Imports

In [1]:

import tensorflow as tf
import os
import sys


from preprocess import create_data_generators, get_class_map

# import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau,ModelCheckpoint

from pathlib import Path

## Constants


In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

## Paths

In [3]:
from pathlib import Path
import os

PROJECT_BASE_DIR = Path(os.environ["NEURODETECT_BASE_DIR"])

TRAIN_DIR = str(PROJECT_BASE_DIR / "train")
TEST_DIR = str(PROJECT_BASE_DIR / "test")
MODELS_DIR = PROJECT_BASE_DIR / "models"

print("Project base:", PROJECT_BASE_DIR)
print("Train path:", TRAIN_DIR)
print("Test path:", TEST_DIR)
print("Models path:", MODELS_DIR)

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists:", os.path.exists(TEST_DIR))
print("Models exists:", os.path.exists(MODELS_DIR))

Project base: G:\My Drive\SeniorProject\datasets\dataset
Train path: G:\My Drive\SeniorProject\datasets\dataset\train
Test path: G:\My Drive\SeniorProject\datasets\dataset\test
Models path: G:\My Drive\SeniorProject\datasets\dataset\models
Train exists: True
Test exists: True
Models exists: True


## Train Gen

In [4]:
train_generator, test_generator = create_data_generators(
    TRAIN_DIR,
    TEST_DIR,
    IMG_SIZE,
    BATCH_SIZE
)

labels = get_class_map(train_generator)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


## Batch

In [5]:
X_batch, y_batch = next(train_generator) 
print("Batch image shape:", X_batch.shape)  

print("\nLoading Test Data:")

X_batch, y_batch = next(test_generator)  
print("Batch image shape:", X_batch.shape) 

Batch image shape: (32, 224, 224, 3)

Loading Test Data:
Batch image shape: (32, 224, 224, 3)


## Baseline Model

In [6]:
num_classes = len(labels)

model_baseline = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_baseline.summary()

d:\SE\SP\SeniorProject\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

## Compile

In [7]:
model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Run Epochs

In [8]:
import time
import math

steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

start = time.time()
history_baseline = model_baseline.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS
)
train_time_sec = time.time() - start

print("Training time (sec):", round(train_time_sec, 2))

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 227s 3s/step - accuracy: 0.4512 - loss: 1.2056 - val_accuracy: 0.3274 - val_loss: 1.9327
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 255s 3s/step - accuracy: 0.5927 - loss: 0.9605 - val_accuracy: 0.3249 - val_loss: 2.3827
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 243s 3s/step - accuracy: 0.6362 - loss: 0.8648 - val_accuracy: 0.4010 - val_loss: 2.5614
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 244s 3s/step - accuracy: 0.6547 - loss: 0.8260 - val_accuracy: 0.3579 - val_loss: 3.2161
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 245s 3s/step - accuracy: 0.6338 - loss: 0.8281 - val_accuracy: 0.4137 - val_loss: 2.3545
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 258s 3s/step - accuracy: 0.6711 - loss: 0.7754 - val_accuracy: 0.4086 - val_loss: 2.4819
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 231s 3s/step - accuracy: 0.6791 - loss: 0.7424 - val_accuracy: 0.4086 - val_loss: 2.5909
Epoch 8/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 175s 2s/step - accuracy: 0.6990 - loss: 0.7324 - val_accuracy: 0.3655 - v

## Results

In [9]:
baseline_loss, baseline_acc = model_baseline.evaluate(
    test_generator,
    steps=validation_steps
)
print("Baseline Loss:", baseline_loss)
print("Baseline Accuracy:", baseline_acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 841ms/step - accuracy: 0.4975 - loss: 2.6018
Baseline Loss: 2.6018073558807373
Baseline Accuracy: 0.4974619150161743


## Analysis

In [10]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# reset generator to start
test_generator.reset()

y_prob = model_baseline.predict(test_generator, steps=validation_steps)
y_pred = np.argmax(y_prob, axis=1)

y_true = test_generator.classes  # integer labels in directory order
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 824ms/step
Confusion Matrix:
 [[ 17  16  62   5]
 [  2  41  66   6]
 [  2   2 101   0]
 [  5   6  26  37]]

Classification Report:

                  precision    recall  f1-score   support

    glioma_tumor       0.65      0.17      0.27       100
meningioma_tumor       0.63      0.36      0.46       115
        no_tumor       0.40      0.96      0.56       105
 pituitary_tumor       0.77      0.50      0.61        74

        accuracy                           0.50       394
       macro avg       0.61      0.50      0.47       394
    weighted avg       0.60      0.50      0.46       394



## Save Model

In [11]:
os.makedirs(MODELS_DIR, exist_ok=True)

baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
model_baseline.save(str(baseline_model_path))
print(f"Saved to {baseline_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\baseline_cnn.keras


# Transfer Model

#### Phase 1

In [12]:

transfer_train_datagen= ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85,1.15],
    fill_mode='nearest'
)
transfer_test_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_gen_tf=transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
test_gen_tf=transfer_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


## Model

In [13]:
base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

X= base_model.output
X=GlobalAveragePooling2D()(X)
X=Dense(256, activation='relu')(X)
X=Dropout(0.4)(X)
X=Dense(128,activation='relu')(X)
X=Dropout(0.3)(X)
outputs= Dense(train_gen_tf.num_classes, activation='softmax')(X)
model_transfer=Model(inputs=base_model.input, outputs=outputs)

## Compile

In [14]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1=[
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_1r=1e-6),
    ModelCheckpoint(
        str(MODELS_DIR / 'hopefully_best_transfer_phase1.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

## Epochs

In [15]:
EPOCHS_TL=20
history_transfer= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=EPOCHS_TL,
    callbacks=callbacks_phase1
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 259s 3s/step - accuracy: 0.6439 - loss: 0.8868 - val_accuracy: 0.4975 - val_loss: 1.3042 - learning_rate: 0.0010
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 163s 2s/step - accuracy: 0.7763 - loss: 0.5606 - val_accuracy: 0.5990 - val_loss: 1.2496 - learning_rate: 0.0010
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 368s 4s/step - accuracy: 0.8028 - loss: 0.4947 - val_accuracy: 0.6218 - val_loss: 1.3660 - learning_rate: 0.0010
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 248s 3s/step - accuracy: 0.8303 - loss: 0.4466 - val_accuracy: 0.5990 - val_loss: 1.2997 - learning_rate: 0.0010
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 235s 3s/step - accuracy: 0.8390 - loss: 0.4183 - val_accuracy: 0.6294 - val_loss: 1.2543 - learning_rate: 0.0010
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 223s 2s/step - accuracy: 0.8519 - loss: 0.3706 - val_accuracy: 0.6701 - val_loss: 1.4312 - learning_rate: 5.0000e-04
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.8732 - loss: 0.3329 - val

## Results

In [16]:
transfer_loss, transfer_acc = model_transfer.evaluate(test_gen_tf)
print("Transfer Learning Loss:", transfer_loss)

13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 630ms/step - accuracy: 0.7259 - loss: 1.3478
Transfer Learning Loss: 1.3477507829666138


## Comparision

In [17]:
# print('Baseline Accuracy:', baseline_acc)
print('Transfer Learning Accuracy:', transfer_acc)

Transfer Learning Accuracy: 0.7258883118629456


In [18]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
historical_finetune= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=10
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - accuracy: 0.6355 - loss: 1.2100 - val_accuracy: 0.6827 - val_loss: 1.7032
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8105 - loss: 0.5276 - val_accuracy: 0.6701 - val_loss: 1.8394
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.8463 - loss: 0.4331 - val_accuracy: 0.6523 - val_loss: 1.9215
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.8585 - loss: 0.3696 - val_accuracy: 0.6574 - val_loss: 1.8780
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.8686 - loss: 0.3466 - val_accuracy: 0.6599 - val_loss: 1.8494
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.8770 - loss: 0.3304 - val_accuracy: 0.6726 - val_loss: 1.6875
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.8843 - loss: 0.3083 - val_accuracy: 0.6954 - val_loss: 1.6200
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.8899 - loss: 0.3007 - val_accuracy: 0.7005 - va

## Save Model

In [19]:
os.makedirs(MODELS_DIR, exist_ok=True)

transfer_model_path = MODELS_DIR / "mobilenetv2_transfer.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_transfer.keras


#### Phase 2

### Callbacks

In [20]:
callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint(
        str(MODELS_DIR / 'best_transfer_phase2.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

### Unfreeze + Recompile

In [21]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### Run

In [ ]:
history_finetune = model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=20,
    callbacks=callbacks_phase2
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.9031 - loss: 0.2567 - val_accuracy: 0.7107 - val_loss: 1.4214 - learning_rate: 1.0000e-05
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - accuracy: 0.9021 - loss: 0.2440 - val_accuracy: 0.7157 - val_loss: 1.3890 - learning_rate: 1.0000e-05
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.9139 - loss: 0.2298 - val_accuracy: 0.7234 - val_loss: 1.3491 - learning_rate: 1.0000e-05
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.9139 - loss: 0.2365 - val_accuracy: 0.7284 - val_loss: 1.2807 - learning_rate: 1.0000e-05
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.9202 - loss: 0.2180 - val_accuracy: 0.7259 - val_loss: 1.2337 - learning_rate: 1.0000e-05
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 97s 1s/step - accuracy: 0.9220 - loss: 0.2109 - val_accuracy: 0.7234 - val_loss: 1.2044 - learning_rate: 1.0000e-05
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.9233 - 

### Compare

In [ ]:
test_gen_tf.reset()
finetune_loss, finetune_acc = model_transfer.evaluate(test_gen_tf)
print(f"Phase 1 Accuracy:  {transfer_acc:.4f}")
print(f"Phase 2 Accuracy:  {finetune_acc:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 551ms/step - accuracy: 0.7868 - loss: 1.2671
Phase 1 Accuracy:  0.7411
Phase 2 Accuracy:  0.7868


### Save

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
transfer_model_path = MODELS_DIR / "mobilenetv2_finetuned.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_finetuned.keras


# Phase 3

## Callbacks

In [ ]:
callbacks_phase3 = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-8),
    ModelCheckpoint(
        str(MODELS_DIR / 'best_transfer_phase3.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

## Unfreeze more Layers 

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-60]: 
    layer.trainable = False

model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Run Epochs

In [ ]:
history_phase3 = model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=30,  
    callbacks=callbacks_phase3
)

Epoch 1/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.9537 - loss: 0.1283 - val_accuracy: 0.7792 - val_loss: 1.2669 - learning_rate: 5.0000e-06
Epoch 2/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.9578 - loss: 0.1200 - val_accuracy: 0.7817 - val_loss: 1.2588 - learning_rate: 5.0000e-06
Epoch 3/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9575 - loss: 0.1084 - val_accuracy: 0.7868 - val_loss: 1.2198 - learning_rate: 5.0000e-06
Epoch 4/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.9519 - loss: 0.1303 - val_accuracy: 0.7665 - val_loss: 1.2983 - learning_rate: 5.0000e-06
Epoch 5/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9624 - loss: 0.1089 - val_accuracy: 0.7766 - val_loss: 1.3052 - learning_rate: 5.0000e-06
Epoch 6/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.9568 - loss: 0.1299 - val_accuracy: 0.7589 - val_loss: 1.3428 - learning_rate: 5.0000e-06
Epoch 7/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9568 

## Evaluate Improvements

In [ ]:
test_gen_tf.reset()
phase3_loss, phase3_acc = model_transfer.evaluate(test_gen_tf)
print(f"Phase 1 Accuracy: {transfer_acc:.4f}")
print(f"Phase 2 Accuracy: {finetune_acc:.4f}")
print(f"Phase 3 Accuracy: {phase3_acc:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 572ms/step - accuracy: 0.7868 - loss: 1.2198
Phase 1 Accuracy: 0.7411
Phase 2 Accuracy: 0.7868
Phase 3 Accuracy: 0.7868
